# 🔍 Evaluator-Optimizer: Generate, Critique, Refine

The **evaluator-optimizer** workflow generates a candidate, grades it against
explicit criteria, and feeds the critique back into a regeneration — looping
until the grade passes.

This notebook rebuilds the joke generator from
`01_evaluator_optimizer_langgraph.ipynb` in this module — same
evaluation schema and criteria, same topic, same iteration cap — twice: once as a plain LCEL loop,
and once with LangChain 1.x agent middleware, which is the most natural home this
pattern has in the 1.x API.

```
topic ──▶ generate ──▶ evaluate ──┬── approved ──────────▶ final
             ▲                    │
             └── feedback ────────┘  (bounded, or it never ends)
```

## Learning Objectives
In this notebook, you will learn:
1. **The cycle problem** - why a pure LCEL chain cannot loop back on itself
2. **Structured grading** - per-criterion judgements plus actionable feedback, with the verdict computed in code
3. **The LCEL loop** - drive the cycle from Python and keep the chains declarative
4. **Middleware loops** - use `after_model` with `jump_to="model"` for a native 1.x cycle
5. **Bounding the loop** - why `ModelCallLimitMiddleware` is not optional

## Where this fits
`01_evaluator_optimizer_langgraph.ipynb` built this loop as a graph. The LCEL notebooks of modules 05–08 each found one operator per pattern; this one shows why a loop needs something else.

## Prerequisites
- The pattern and its evaluation schema — `01_evaluator_optimizer_langgraph.ipynb`
- `create_agent` and middleware — `modules/04_langchain_agents_and_middleware/notebooks/04_middleware.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📦 Step 1: Environment Setup

Load the API key and create the generator and evaluator models through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys & Import Dependencies
# ============================================================================
# We use python-dotenv to securely load API keys from a .env file
# This is a best practice - never hardcode API keys in your notebooks!
# ============================================================================

import warnings

from aied_helpers import get_openai_llm
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, after_model
from langchain.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# temperature=0.7 — the generator needs room to vary between drafts, otherwise
# every refinement pass returns the same joke and the loop cannot converge.
llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

# temperature=0 for the evaluator: the same draft should get the same grade.
evaluator_llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print("✅ Environment variables loaded successfully!")
print(f"🤖 Generator: {llm.model_name} (temperature={llm.temperature}), evaluator temperature={evaluator_llm.temperature}")

---

## 🔁 Part 1: The Cycle Problem

Every pattern so far mapped onto an LCEL operator because every pattern was a
**directed acyclic** flow. This one is not.

`a | b | a` does not loop. It builds a three-step chain that runs `a` twice. LCEL
has no operator that says "go back and try again, conditionally, until a
predicate passes". The `.with_retry()` trick from `modules/05_prompt_chaining/notebooks/03_prompt_chaining_lcel.ipynb` came close, but it
retries with **no memory of why it failed** — and this pattern's whole point is
feeding the critique forward.

So there are two honest options, and we build both:

| Approach | The loop lives in | Use when |
|---|---|---|
| LCEL plus a Python `while` | your driver function | the loop is orchestration you own |
| `create_agent` plus `after_model` | the agent graph, via `jump_to` | you want a single invocable object |

---

## 📋 Part 2: The Evaluation Schema

Identical to the LangGraph version: one boolean per criterion plus feedback, a
word-count check in code, and a verdict **computed** from them. The verdict is a
plain string so it can be tested with `==`, and the feedback is kept separate so
it can be interpolated into the next generation prompt.

In [ ]:
# ============================================================================
# EVALUATION SCHEMA: Per-criterion judgement + a verdict computed in code
# ============================================================================


class JokeReview(BaseModel):
    """One pass/fail judgement per criterion, plus a fix for each failure."""

    clear_punchline: bool = Field(
        description="True if the joke has a setup followed by a distinct punchline that lands the joke"
    )
    topic_specific: bool = Field(
        description=(
            "True if the punchline depends on a concrete feature of agentic AI "
            "(tool calls, planning, memory, loops, autonomy), not a generic computer pun"
        )
    )
    avoids_stock_template: bool = Field(
        description="True if it does NOT use a stock template like 'Why did the ...?', 'X walks into a bar' or 'What do you call ...?'"
    )
    appropriate: bool = Field(description="True if it is appropriate and inoffensive")
    improvement_feedback: str = Field(
        description="For each criterion that is False, a concrete fix. Empty if all pass."
    )


MAX_WORDS = 40  # checked in code: an LLM should not be asked to count


def verdict(review: JokeReview, joke: str) -> tuple[str, str]:
    """Combine the LLM's per-criterion judgement with the code check into one grade."""
    short_enough = len(joke.split()) <= MAX_WORDS
    passed = all(
        [review.clear_punchline, review.topic_specific, review.avoids_stock_template, review.appropriate, short_enough]
    )
    feedback = review.improvement_feedback
    if not short_enough:
        feedback += f" Cut it to at most {MAX_WORDS} words."
    return ("approved" if passed else "needs_improvement"), feedback.strip()


evaluator = ChatPromptTemplate.from_template(
    "Review this joke about agentic AI systems. Judge each criterion independently; "
    "a criterion that is met passes even if the joke could be better.\n\nJoke: {joke}"
) | evaluator_llm.with_structured_output(JokeReview)

print("✅ Evaluator ready")

---

## ✍️ Part 3: The Generator

One chain, two prompts. The first pass has no feedback to work with; every later
pass gets the previous draft and the feedback, so it revises rather than starts over. Keeping this as a single chain with an optional `{feedback_block}` slot is
simpler than maintaining two near-identical chains.

In [ ]:
# ============================================================================
# GENERATOR CHAIN: Writes a joke, optionally incorporating feedback
# ============================================================================
generator = ChatPromptTemplate.from_template(
    "Create a well-crafted, humorous joke about {topic}. Reply with the joke only.\n\n"
    "{feedback_block}"
) | llm | StrOutputParser()


def feedback_block(previous: str | None, feedback: str | None) -> str:
    """Render the revision section of the prompt, or nothing on the first pass."""
    if not feedback:
        return ""
    return (
        f"Here is your previous joke:\n{previous}\n\n"
        f"Rewrite it, incorporating this feedback from a reviewer:\n{feedback}\n\n"
        "Make sure to address the specific points mentioned."
    )


print("✅ Generator ready")

---

## 🐍 Part 4: Approach A: The LCEL Loop

The chains stay declarative; a small driver function owns the cycle. This is the
most direct translation of the LangGraph graph, and it makes the loop bound
impossible to overlook.

> **Note**: the cap is not optional — a stubborn evaluator would loop forever.
> `01_evaluator_optimizer_langgraph.ipynb` checks it in the router; here the
> driver's `range(MAX_ITERATIONS)` is the cap.

In [ ]:
# ============================================================================
# LCEL LOOP: Generate and evaluate until approved, or the cap is reached
# ============================================================================
MAX_ITERATIONS = 5


def run_evaluator_optimizer(topic: str) -> dict:
    """Generate, critique and refine until the evaluator approves."""
    print(f"Starting evaluator-optimizer workflow for topic: {topic}")
    feedback = None
    joke = None

    for iteration in range(1, MAX_ITERATIONS + 1):
        joke = generator.invoke({"topic": topic, "feedback_block": feedback_block(joke, feedback)})
        grade, feedback = verdict(evaluator.invoke({"joke": joke}), joke)

        print(f"\n🔄 Iteration {iteration}: {grade}")
        print(f"   Draft: {joke[:100]}...")

        if grade == "approved":
            print(f"✅ Approved after {iteration} iteration(s)")
            return {
                "content_topic": topic,
                "generated_content": joke,
                "quality_assessment": grade,
                "iterations": iteration,
            }

        print(f"   Feedback: {feedback[:120]}...")

    print(f"⚠️  Stopped at the {MAX_ITERATIONS}-iteration cap without approval")
    return {
        "content_topic": topic,
        "generated_content": joke,
        "quality_assessment": grade,
        "iterations": MAX_ITERATIONS,
    }

In [ ]:
# ============================================================================
# TEST A: Run the LCEL loop on the same topic as the LangGraph notebook
# ============================================================================
result = run_evaluator_optimizer("Agentic AI systems")

print("\n" + "=" * 50)
print("EVALUATOR-OPTIMIZER WORKFLOW RESULTS")
print("=" * 50)
print(f"Topic: {result['content_topic']}")
print(f"Iterations: {result['iterations']}")
print(f"Quality Status: {result['quality_assessment']}")
print(f"\nFinal Content:\n{result['generated_content']}")

---

## 🧩 Part 5: Approach B: The Middleware Loop

LangChain 1.x can express the cycle **inside** a single agent. The `after_model`
hook runs once the model has answered, and returning `jump_to="model"` sends
control back for another pass. That is a real conditional edge in the agent
graph, not a Python loop around it.

Three details make it work:

- `can_jump_to=["model"]` must be declared on the decorator. It is what creates
  the conditional edge; without it the jump is ignored.
- Appending a `HumanMessage` with the critique is how the feedback reaches the
  next generation. The message history **is** the state.
- Returning `None` means "no jump" — the agent falls through and finishes.

In [ ]:
# ============================================================================
# MIDDLEWARE: Evaluate after each model call, loop back when not approved
# ============================================================================
review_log: list[str] = []


@after_model(can_jump_to=["model"])
def evaluate_and_refine(state, runtime):
    """Grade the latest draft; jump back to the model when it needs work."""
    last = state["messages"][-1]

    # Only grade a finished assistant answer, never a tool-calling turn.
    if not isinstance(last, AIMessage) or getattr(last, "tool_calls", None):
        return None

    draft = last.text if hasattr(last, "text") else str(last.content)
    grade, feedback = verdict(evaluator.invoke({"joke": draft}), draft)
    review_log.append(f"{grade}: {draft[:60]}...")

    if grade == "approved":
        print(f"✅ Approved: {draft[:80]}...")
        return None  # no jump — the agent ends here

    print(f"🔄 Needs improvement: {feedback[:100]}...")
    return {
        "messages": [
            HumanMessage(
                content=(
                    f"A reviewer graded that joke as needing improvement.\n"
                    f"Feedback: {feedback}\n\n"
                    f"Rewrite the joke addressing this feedback."
                )
            )
        ],
        "jump_to": "model",
    }


print("✅ Middleware defined")

The call-limit middleware is the bound. Without it a never-satisfied evaluator
loops until you kill the kernel. `exit_behavior="end"` stops cleanly and returns
the last draft rather than raising.

In [ ]:
# ============================================================================
# AGENT: Generator plus evaluator loop plus a hard iteration cap
# ============================================================================
joke_agent = create_agent(
    model=llm,
    tools=[],  # no tools — the loop is the whole mechanism
    system_prompt="You are a comedy writer. Write one well-crafted, clean joke on the given topic. Reply with the joke only.",
    middleware=[
        evaluate_and_refine,
        ModelCallLimitMiddleware(run_limit=MAX_ITERATIONS, exit_behavior="end"),
    ],
)

print("✅ Agent assembled")

In [ ]:
# ============================================================================
# TEST B: Same topic, same criteria, run through the agent
# ============================================================================
review_log.clear()

response = joke_agent.invoke(
    {"messages": [HumanMessage(content="Write a joke about Agentic AI systems")]}
)

print("\n" + "=" * 50)
print("MIDDLEWARE LOOP RESULTS")
print("=" * 50)
print(f"Model calls: {len(review_log)}")
for entry in review_log:
    print(f"  - {entry}")
print(f"\nFinal Content:\n{response['messages'][-1].text}")

---

## ⚖️ Part 6: Choosing Between the Two

Both loops are bounded, both feed critique forward, and both produce the same
result. They differ in what you get around the loop.

| | LCEL loop | Middleware loop |
|---|---|---|
| The loop is | a Python `for` you can read at a glance | a conditional edge inside the agent |
| Bounded by | your `range(MAX_ITERATIONS)` | `ModelCallLimitMiddleware` |
| Result is | a plain dict you shaped | an agent state with full message history |
| Tracing | two separate chain runs per pass | one agent run, all passes nested |
| Adding tools | you would rebuild the loop by hand | already supported — the agent has a tool node |
| Streaming, checkpointing, interrupts | none | inherited from the agent runtime |

Start with the LCEL loop when the generator is a single call and the loop is
orchestration you own. Move to the middleware loop when the generator needs
tools, or when you want one invocable object with the agent runtime's streaming
and persistence.

---

## 📝 Summary

We rebuilt the LangGraph evaluator-optimizer twice, and this is the one pattern
where LCEL alone is not enough.

### 1. The translation table

| LangGraph | LangChain 1.x |
|---|---|
| `add_conditional_edges` back to `generate_content` | a Python loop, **or** `after_model` returning `jump_to="model"` |
| `WorkflowState` carrying feedback between passes | a local variable, **or** a `HumanMessage` appended to history |
| `with_structured_output(JokeReview)` + `verdict()` | identical — unchanged in 1.x |
| `MAX_ITERATIONS` checked in the router | `range(MAX_ITERATIONS)`, **or** `ModelCallLimitMiddleware(run_limit=N)` |

### 2. What to remember
- **LCEL is acyclic.** `a | b | a` runs `a` twice, it does not loop. Cycles come
  from a Python loop, from agent middleware, or from LangGraph.
- **`.with_retry()` is not this pattern.** It retries blind. Evaluator-optimizer
  exists to feed the critique forward.
- **Always bound the loop.** A structured verdict does not guarantee the
  evaluator will ever say "approved".
- **`can_jump_to` is mandatory** on the decorator. It is what builds the
  conditional edge.

### 3. The five patterns, complete

| Pattern | LangChain 1.x primitive | Full coverage |
|---|---|---|
| Prompt chaining | `\|` plus `.with_retry()` plus `.with_fallbacks()` | yes |
| Routing | `RunnableBranch` | yes |
| Parallelization | `RunnableParallel` | yes |
| Orchestrator-worker | `.map()` | fan-out yes; shared state and per-worker durability need `Send` |
| Evaluator-optimizer | Python loop, or `after_model` plus `jump_to` | yes |

### Next Steps
- **`modules/10_planning/notebooks/01_planning_react_vs_plan.ipynb`**: from fixed workflows to agents that plan their own steps
- Read [`docs/workflow-patterns/langchain-implementations.md`](../../../../docs/source/06/workflow-patterns/langchain-implementations.md) for the full LangGraph-to-LangChain mapping
- For cycles that need checkpointing, interrupts or human-in-the-loop, see
  [repository 04 — LangGraph fundamentals](https://github.com/Sourav692/ai-engineering-demystified-04-langgraph-fundamentals)